# WAV2HYP: Catalog comparer

Compare two **locator** H5 catalogs (Canonical vs Test): event rates, map overlay, origin offsets (polar), and shared-arrival timing.

**Y metric for arrival scatter:** The documented `arrivals_table` schema does not include EQT `peak_value`. The helper uses **`peak_value_*` if those columns exist** on your file; otherwise it falls back to NLL **`weight`** or **`residual`**, with axis labels updated automatically.

Edit `CONFIG`, then run top-to-bottom. Figures are **displayed** and **saved** under `notebooks/<volcano>/`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import yaml
from IPython.display import display

from wav2hyp.config_loader import config_path_anchor
from wav2hyp.viz import catalog_comparer as cc
from wav2hyp.viz.catalog_explorer_utils import load_notebook_context, print_context_summary
from wav2hyp.viz.notebook_styles import apply_notebook_style

apply_notebook_style()

## Setup

- `locator_h5_test`: set to an alternate `nll.h5`, or leave unset to use the default from the YAML (`output.base_dir` / `output.locator_dir` / `nll.h5`).
- `histogram_freq`: pandas offset string for the event-rate plot (e.g. `1h`).

In [ ]:
CONFIG = {
    "config_file": "examples_local/sthelens.yaml",
    "run_base_dir": None,
    "locator_h5_canonical": None,
    "locator_h5_test": None,
    "label_canonical": "Canonical",
    "label_test": "Test",
    "t1": "2004-09-23T00:00:00",
    "t2": "2004-09-24T00:00:00",
    "event_match_tolerance_s": 5.0,
    "histogram_freq": "1h",
}

In [ ]:
CONFIG_FILE = CONFIG.get("config_file", "examples_local/sthelens.yaml")

_cwd = Path.cwd()
if (_cwd / CONFIG_FILE).is_file():
    CONFIG_PATH = (_cwd / CONFIG_FILE).resolve()
elif (_cwd.parent / CONFIG_FILE).is_file():
    CONFIG_PATH = (_cwd.parent / CONFIG_FILE).resolve()
else:
    CONFIG_PATH = (_cwd / CONFIG_FILE).resolve()

with open(CONFIG_PATH) as f:
    _YAML_CFG = yaml.safe_load(f)

if CONFIG.get("run_base_dir"):
    RUN_BASE_DIR = Path(CONFIG["run_base_dir"]).expanduser().resolve()
else:
    RUN_BASE_DIR = (config_path_anchor(CONFIG_PATH) / _YAML_CFG["output"]["base_dir"]).resolve()

_locator_sub = _YAML_CFG["output"].get("locator_dir", "locations")
_DEFAULT_NLL_H5 = (RUN_BASE_DIR / _locator_sub / "nll.h5").resolve()

LOCATOR_H5_CANONICAL = (
    Path(CONFIG["locator_h5_canonical"]).expanduser().resolve()
    if CONFIG.get("locator_h5_canonical")
    else _DEFAULT_NLL_H5
)
if not CONFIG.get("locator_h5_test"):
    LOCATOR_H5_TEST = _DEFAULT_NLL_H5
else:
    LOCATOR_H5_TEST = Path(CONFIG["locator_h5_test"]).expanduser().resolve()

LABEL_CANONICAL = str(CONFIG.get("label_canonical", "Canonical"))
LABEL_TEST = str(CONFIG.get("label_test", "Test"))
T1 = CONFIG.get("t1")
T2 = CONFIG.get("t2")
EVENT_MATCH_TOLERANCE_S = float(CONFIG.get("event_match_tolerance_s", 5.0))
HISTOGRAM_FREQ = str(CONFIG.get("histogram_freq", "1h"))

NOTEBOOK_OUT = cc.default_notebook_export_dir(CONFIG_PATH)
VOLCANO_SLUG = CONFIG_PATH.stem
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

ctx = load_notebook_context(
    config_path=CONFIG_PATH,
    volcano_slug=VOLCANO_SLUG,
    notebook_output_dir=NOTEBOOK_OUT,
    run_base_dir=CONFIG.get("run_base_dir"),
    locator_h5=LOCATOR_H5_CANONICAL,
)
print_context_summary(ctx)
print("NOTEBOOK_OUT          ", NOTEBOOK_OUT)
print("LOCATOR_H5_CANONICAL  ", LOCATOR_H5_CANONICAL, "exists:", LOCATOR_H5_CANONICAL.exists())
print("LOCATOR_H5_TEST      ", LOCATOR_H5_TEST, "exists:", LOCATOR_H5_TEST.exists())
print("T1/T2                ", T1, T2, "| histogram", HISTOGRAM_FREQ)


In [ ]:
catalog_canonical_df, arrivals_canonical_df = cc.load_locator_tables(
    LOCATOR_H5_CANONICAL, t1=T1, t2=T2
)
catalog_test_df, arrivals_test_df = cc.load_locator_tables(LOCATOR_H5_TEST, t1=T1, t2=T2)

print(f"{LABEL_CANONICAL}: events={len(catalog_canonical_df):,}, arrivals={len(arrivals_canonical_df):,}")
print(f"{LABEL_TEST}: events={len(catalog_test_df):,}, arrivals={len(arrivals_test_df):,}")
display(catalog_canonical_df.head(3))
display(catalog_test_df.head(3))

## Partitions: Canonical only, Test only, Both

Matching: nearest `origin_time` in window, then one-to-one de-duplication (see `cc.match_events_one_to_one`).

In [ ]:
matched_pairs = cc.match_events_one_to_one(
    catalog_canonical_df,
    catalog_test_df,
    tolerance_s=EVENT_MATCH_TOLERANCE_S,
)
matched_loc = cc.enrich_matched_pairs_locations(
    matched_pairs, catalog_canonical_df, catalog_test_df
)
canonical_only_df, test_only_df, both_at_canonical_df = cc.partition_by_match(
    catalog_canonical_df, catalog_test_df, matched_pairs
)

print(
    f"matched pairs: {len(matched_pairs):,} | canonical-only: {len(canonical_only_df):,} | test-only: {len(test_only_df):,}"
)
display(matched_loc.head(8))

In [ ]:
shared_arrivals = cc.build_shared_arrivals(
    matched_pairs, arrivals_canonical_df, arrivals_test_df
)
print(f"shared arrivals rows: {len(shared_arrivals):,}")
display(shared_arrivals.head(12))

## Plots (inline + `notebooks/<volcano>/`)


In [ ]:
fig_rate = cc.plot_timebin_stairs(
    catalog_canonical_df,
    catalog_test_df,
    freq=HISTOGRAM_FREQ,
    label_canonical=LABEL_CANONICAL,
    label_test=LABEL_TEST,
)
display(fig_rate)
cc.save_figure(fig_rate, NOTEBOOK_OUT, cc.FIG_NAME_TIMEBIN, volcano_slug=VOLCANO_SLUG)


In [ ]:
fig_pol = cc.plot_polar_origin_offset(matched_loc, title=None)
display(fig_pol)
cc.save_figure(fig_pol, NOTEBOOK_OUT, cc.FIG_NAME_POLAR, volcano_slug=VOLCANO_SLUG)


In [ ]:
vfig = cc.make_catalog_comparer_volcano_figure(
    ctx.cfg,
    ctx.inventory_path,
    canonical_only_df,
    test_only_df,
    both_at_canonical_df,
    map_t1=str(T1) if T1 else None,
    map_t2=str(T2) if T2 else None,
)
display(vfig)
cc.save_figure(vfig, NOTEBOOK_OUT, cc.FIG_NAME_VOLCANO, volcano_slug=VOLCANO_SLUG)


In [ ]:
if len(shared_arrivals) == 0:
    print("no shared arrivals — skip arrival scatter / boxplot")
else:
    fig_s = cc.plot_arrival_scatter_p_s_panels(shared_arrivals)
    display(fig_s)
    cc.save_figure(fig_s, NOTEBOOK_OUT, cc.FIG_NAME_ARRIVAL_SCATTER, volcano_slug=VOLCANO_SLUG)


In [ ]:
if len(shared_arrivals) == 0:
    pass
else:
    fig_b = cc.plot_arrival_boxplot_by_station(shared_arrivals)
    display(fig_b)
    cc.save_figure(fig_b, NOTEBOOK_OUT, cc.FIG_NAME_ARRIVAL_BOXPLOT, volcano_slug=VOLCANO_SLUG)
